# 🔮 03. Reset-Aware Fill & Physical Demand Prediction

An inspectable baseline: median completed-cycle rate × elapsed days since last emptying, capped at nominal 100%. No previous pre-emptying percentage is added. This is not a calibrated overflow-probability model.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "Zakaria-Work"
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from Zakaria-Work or its parent directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loader import load_and_clean_data, collection_summary
from src.clustering import extract_container_profiles, cluster_containers
from src.predictor import predict_container_demands, select_collection_sites
OUT = ROOT / "data" / "corrected"
OUT.mkdir(parents=True, exist_ok=True)
MAT_COLORS = {"white": "#1f77b4", "green": "#2ca02c", "brown": "#8c564b"}
plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 100})


## 1. Build leakage-safe profiles
Rebuild the same dispatch-date prefix instead of loading stale profiles from another date.

In [ ]:
df = load_and_clean_data(ROOT / "glass_fill_levels_anonymized_f.csv")
print(f"{len(df):,} pre-emptying observations; {df.Location_id.nunique()} locations; "
      f"{len(df.groupby(['Location_id', 'material_type']))} observed bins")
print(f"Coverage: {df.date.min().date()} to {df.date.max().date()}")
display(df.head())

# Change this explicit morning dispatch date to replay a different day.
PLANNING_DATE = pd.Timestamp("2026-08-17")
profiles = extract_container_profiles(df, as_of=PLANNING_DATE, inclusive=False)
clustered_df, kmeans_model = cluster_containers(profiles, n_clusters=4)
print(f"Known bins strictly before dispatch: {len(profiles)}")


## 2. Forecast at one common target date
Default target is the morning dispatch date. The neutral holiday multiplier is 1.0; any changed multiplier is a user scenario assumption. Unknown rates remain unknown. Container volume is 3 m³; 350 kg/m³ is an explicit assumed density.

In [ ]:
demands = predict_container_demands(clustered_df, planning_date=PLANNING_DATE,
    days_ahead=0, holiday_multiplier=1.0, container_volume_m3=3.0, glass_density_kg_m3=350.0)
display(demands[['Location_id','material_type','last_reading_date','last_pre_emptying_fill_perc',
                 'predicted_fill_perc','rate_source','urgency_status']].head(10))
print("Forecast at dispatch. Set days_ahead for an uncollected future scenario, not an observed future value.")


## 3. Urgency bands and site demand
Urgent ≥80%, opportunistic ≥60%, otherwise safe. At an urgent site, eligible neighboring materials can share the stop. Thresholds are configurable planning assumptions; they do not establish service guarantees.

In [ ]:
sites, decisions = select_collection_sites(demands)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
demands.urgency_status.value_counts().plot.barh(ax=axes[0], color='#4676a3')
axes[0].set(title='Predicted dispatch bands', xlabel='Bins')
demands.loc[demands.forecast_available, 'demand_weight_kg'].plot.hist(ax=axes[1], bins=25)
axes[1].set(title='Estimated bin collection mass', xlabel='kg (assumed bulk density)')
plt.tight_layout()
plt.show()
print(f'{len(sites)} selected sites / {decisions.selected.sum()} selected bins; all skips retain reasons.')
display(decisions.selection_reason.value_counts())


## 4. Export and evaluate
Evaluation is rolling and chronological at actual visits. Numeric errors exclude overfull truths. It does not measure unobserved occupancy or simulated route savings.

In [ ]:
demands.to_csv(OUT / 'predicted_demands.csv', index=False)
decisions.to_csv(OUT / 'bin_decisions.csv', index=False)
sites.to_csv(OUT / 'selected_sites.csv', index=False)
# Run separately for a full chronological report:
# from src.evaluation import evaluate_forecasts
# metrics, cases = evaluate_forecasts(df)
# display(metrics)
